## Part 4: Python and SQL Integration

This section integrates Python with SQL to create a command-line reporting tool. It generates dynamic reports based on the selected date range, including total orders, revenue, unique customers, top products, and previous period comparison.

In [0]:
# ==========================================
# Load Cleaned CSV Files
# ==========================================

orders_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/orders_clean.csv",
    header=True,
    inferSchema=True
)

customers_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/customers_clean.csv",
    header=True,
    inferSchema=True
)

products_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/products_clean.csv",
    header=True,
    inferSchema=True
)

order_items_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/order_items_clean.csv",
    header=True,
    inferSchema=True
)

print("Cleaned CSV files loaded successfully.")

Cleaned CSV files loaded successfully.


In [0]:
# ==========================================
# SQLite Setup
# ==========================================

import sqlite3
from datetime import datetime, timedelta

conn = sqlite3.connect("/tmp/ecommerce.db")

# Spark DataFrames ko SQLite tables me load karo
orders_sql.toPandas().to_sql("orders", conn, if_exists="replace", index=False)
customers_sql.toPandas().to_sql("customers", conn, if_exists="replace", index=False)
products_sql.toPandas().to_sql("products", conn, if_exists="replace", index=False)
order_items_sql.toPandas().to_sql("order_items", conn, if_exists="replace", index=False)

print("SQLite database ready.")

SQLite database ready.


In [0]:
# ==========================================
# Take User Input
# ==========================================

report_type = input(
    "Enter report type (daily/weekly/monthly): "
).lower().strip()

start_date = input(
    "Enter start date (YYYY-MM-DD): "
).strip()

end_date = input(
    "Enter end date (YYYY-MM-DD): "
).strip()

Enter report type (daily/weekly/monthly):  monthly

Enter start date (YYYY-MM-DD):  2025-01-01

Enter end date (YYYY-MM-DD):  2025-12-31

In [0]:
# ==========================================
# Validate Input
# ==========================================

if report_type not in ["daily", "weekly", "monthly"]:
    raise ValueError("Report type must be daily, weekly or monthly.")

start = datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.strptime(end_date, "%Y-%m-%d")

if start > end:
    raise ValueError("Start date cannot be after end date.")

days = (end - start).days + 1

previous_end = start - timedelta(days=1)
previous_start = previous_end - timedelta(days=days - 1)

previous_start = previous_start.strftime("%Y-%m-%d")
previous_end = previous_end.strftime("%Y-%m-%d")

print("Input validated successfully.")

Input validated successfully.


In [0]:
# ==========================================
# Generate Complete Report
# ==========================================

cursor = conn.cursor()

summary_query = """
SELECT
    COUNT(DISTINCT o.order_id),
    COALESCE(ROUND(SUM(
        oi.quantity * oi.unit_price *
        (1 - oi.discount_percent / 100.0)
    ), 2), 0),
    COUNT(DISTINCT o.customer_id)
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
WHERE DATE(o.order_date) BETWEEN ? AND ?
"""

# Current period summary
cursor.execute(summary_query, (start_date, end_date))
total_orders, revenue, unique_customers = cursor.fetchone()

# Previous period summary
cursor.execute(summary_query, (previous_start, previous_end))
_, previous_revenue, _ = cursor.fetchone()

# Percentage change
if previous_revenue == 0:
    percent_change = None
else:
    percent_change = round(
        ((revenue - previous_revenue) / previous_revenue) * 100,
        2
    )

# Top 3 products
top_query = """
SELECT
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS quantity,
    ROUND(SUM(
        oi.quantity * oi.unit_price *
        (1 - oi.discount_percent / 100.0)
    ), 2) AS revenue
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
JOIN products p
ON oi.product_id = p.product_id
WHERE DATE(o.order_date) BETWEEN ? AND ?
GROUP BY p.product_id, p.product_name
ORDER BY revenue DESC
LIMIT 3
"""

cursor.execute(top_query, (start_date, end_date))
top_products = cursor.fetchall()

# Final report
print("\n" + "=" * 40)
print(report_type.upper(), "E-COMMERCE REPORT")
print("=" * 40)

print("Date Range:", start_date, "to", end_date)
print("Total Orders:", total_orders)
print("Total Revenue:", revenue)
print("Unique Customers:", unique_customers)
print("Previous Period Revenue:", previous_revenue)

if percent_change is None:
    print("Percentage Change: Not Available")
else:
    print("Percentage Change:", percent_change, "%")

print("\nTop 3 Products")

if not top_products:
    print("No data found for selected date range.")
else:
    for i, product in enumerate(top_products, 1):
        print(
            i,
            product[0],
            "| Quantity:", product[1],
            "| Revenue:", product[2]
        )


MONTHLY E-COMMERCE REPORT
Date Range: 2025-01-01 to 2025-12-31
Total Orders: 349
Total Revenue: 49131430.49
Unique Customers: 242
Previous Period Revenue: 24549079.36
Percentage Change: 100.14 %

Top 3 Products
1 Deleniti Jeans | Quantity: 21 | Revenue: 639785.44
2 Facere Table | Quantity: 19 | Revenue: 534698.47
3 Dicta Camera | Quantity: 14 | Revenue: 496044.09


In [0]:
# ==========================================
# Close SQLite Connection
# ==========================================

conn.close()

print("SQLite connection closed.")

SQLite connection closed.
